# Implementación funcional de la capa $\theta$

Este notebook implementa y valida la transformación lineal $\theta$ de Keccak.

La capa $\theta$ mezcla las columnas del estado y constituye la primera transformación de cada ronda.

El procedimiento se divide en tres operaciones:

1. cálculo de las paridades de columna;
2. cálculo del efecto de difusión;
3. aplicación del efecto a todo el estado.

En esta etapa se utiliza NumPy para validar la transformación antes de construir su representación mediante restricciones MILP.

## 1. Definición matemática

El estado se representa como:

$$
A[x,y,k],
$$

con:

$$
x,y\in\{0,1,2,3,4\},
$$

y:

$$
k\in\{0,\ldots,z-1\}.
$$

Primero se calcula la paridad de cada columna:

$$
C[x,k]
=
\bigoplus_{y=0}^{4} A[x,y,k].
$$

Después se calcula:

$$
D[x,k]
=
C[x-1 \bmod 5,k]
\oplus
C[x+1 \bmod 5,k-1 \bmod z].
$$

Finalmente:

$$
A_{\theta}[x,y,k]
=
A[x,y,k]
\oplus
D[x,k].
$$

In [1]:
# ============================================================
# CONFIGURACIÓN E IMPORTACIONES
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd

from keccak_milp.layers import (
    column_parities,
    create_single_active_bit_state,
    hamming_weight,
    theta,
    theta_effect,
)


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Python activo    : {sys.executable}")
print(f"NumPy            : {np.__version__}")

Raíz del proyecto: d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada
Python activo    : d:\venvs\keccak-milp\Scripts\python.exe
NumPy            : 2.5.1


## 2. Estado inicial con un solo bit activo

Para observar la difusión producida por $\theta$, se utiliza inicialmente un estado con una única posición activa:

$$
A[2,3,1]=1.
$$

Todas las demás posiciones tienen valor cero.

El peso de Hamming inicial es:

$$
w_H(A)=1.
$$

In [2]:
# ============================================================
# ESTADO CON UN ÚNICO BIT ACTIVO
# ============================================================

z = 4

estado_inicial = create_single_active_bit_state(
    z=z,
    x=2,
    y=3,
    k=1,
)

print("=" * 70)
print("ESTADO INICIAL")
print("=" * 70)
print(f"Forma            : {estado_inicial.shape}")
print(f"Peso de Hamming  : {hamming_weight(estado_inicial)}")
print(
    "Posición activa :",
    np.argwhere(estado_inicial == 1).tolist(),
)
print("=" * 70)

ESTADO INICIAL
Forma            : (5, 5, 4)
Peso de Hamming  : 1
Posición activa : [[2, 3, 1]]


## 3. Paridades de columna

La primera operación calcula:

$$
C[x,k]
=
A[x,0,k]
\oplus
A[x,1,k]
\oplus
A[x,2,k]
\oplus
A[x,3,k]
\oplus
A[x,4,k].
$$

Como existe un único bit activo en:

$$
A[2,3,1],
$$

la única paridad activa debe ser:

$$
C[2,1]=1.
$$

In [3]:
# ============================================================
# CÁLCULO DE LAS PARIDADES
# ============================================================

paridades = column_parities(estado_inicial)

df_paridades = pd.DataFrame(
    paridades,
    index=[f"x={x}" for x in range(5)],
    columns=[f"k={k}" for k in range(z)],
)

print(f"Paridades activas: {np.argwhere(paridades == 1).tolist()}")

df_paridades

Paridades activas: [[2, 1]]


,k=0,k=1,k=2,k=3
x=0,0,0,0,0
x=1,0,0,0,0
x=2,0,1,0,0
x=3,0,0,0,0
x=4,0,0,0,0


## 4. Cálculo del efecto $D$

La matriz $D$ se calcula mediante:

$$
D[x,k]
=
C[x-1 \bmod 5,k]
\oplus
C[x+1 \bmod 5,k-1 \bmod z].
$$

La paridad activa $C[2,1]$ se propaga a dos posiciones de $D$:

$$
D[3,1]=1,
$$

y:

$$
D[1,2]=1.
$$

Cada una de estas posiciones afectará los cinco valores posibles de $y$.

In [4]:
# ============================================================
# CÁLCULO DEL EFECTO D
# ============================================================

efecto = theta_effect(estado_inicial)

df_efecto = pd.DataFrame(
    efecto,
    index=[f"x={x}" for x in range(5)],
    columns=[f"k={k}" for k in range(z)],
)

print(f"Posiciones activas de D: {np.argwhere(efecto == 1).tolist()}")

df_efecto

Posiciones activas de D: [[1, 2], [3, 1]]


,k=0,k=1,k=2,k=3
x=0,0,0,0,0
x=1,0,0,1,0
x=2,0,0,0,0
x=3,0,1,0,0
x=4,0,0,0,0


## 5. Aplicación de $\theta$

Cada valor de $D[x,k]$ se combina con los cinco bits:

$$
A[x,0,k],\ldots,A[x,4,k].
$$

En el ejemplo existen dos posiciones activas en $D$.

Por tanto, se modifican:

$$
2\times5=10
$$

posiciones.

Como la posición inicial no pertenece a esas columnas afectadas, el peso final esperado es:

$$
w_H(A_{\theta})=1+10=11.
$$

In [5]:
# ============================================================
# APLICACIÓN DE THETA
# ============================================================

estado_theta = theta(estado_inicial)

posiciones_theta = np.argwhere(
    estado_theta == 1
).tolist()

print("=" * 70)
print("RESULTADO DE THETA")
print("=" * 70)
print(f"Peso inicial     : {hamming_weight(estado_inicial)}")
print(f"Peso después     : {hamming_weight(estado_theta)}")
print(f"Bits activos     : {len(posiciones_theta)}")
print(f"Posiciones       : {posiciones_theta}")
print("=" * 70)

assert hamming_weight(estado_theta) == 11

RESULTADO DE THETA
Peso inicial     : 1
Peso después     : 11
Bits activos     : 11
Posiciones       : [[1, 0, 2], [1, 1, 2], [1, 2, 2], [1, 3, 2], [1, 4, 2], [2, 3, 1], [3, 0, 1], [3, 1, 1], [3, 2, 1], [3, 3, 1], [3, 4, 1]]


In [6]:
# ============================================================
# TABLA DE POSICIONES ACTIVAS
# ============================================================

df_posiciones_theta = pd.DataFrame(
    posiciones_theta,
    columns=["x", "y", "k"],
)

df_posiciones_theta

,x,y,k
0,1,0,2
1,1,1,2
2,1,2,2
3,1,3,2
4,1,4,2
5,2,3,1
6,3,0,1
7,3,1,1
8,3,2,1
9,3,3,1


## 6. Cancelación por XOR

La operación XOR depende de la paridad, no únicamente de la cantidad de bits activos.

Dos bits activos dentro de la misma columna y posición $k$ producen:

$$
1\oplus1=0.
$$

Por ejemplo:

$$
A[2,1,3]=1,
$$

$$
A[2,4,3]=1.
$$

Entonces:

$$
C[2,3]=0.
$$

Este comportamiento es importante para el modelo MILP, porque no basta con contar bits activos: se debe representar exactamente la paridad.

In [7]:
# ============================================================
# EJEMPLO DE CANCELACIÓN DE PARIDAD
# ============================================================

estado_cancelacion = np.zeros(
    (5, 5, 4),
    dtype=np.int64,
)

estado_cancelacion[2, 1, 3] = 1
estado_cancelacion[2, 4, 3] = 1

paridades_cancelacion = column_parities(
    estado_cancelacion
)

print(f"Peso inicial         : {hamming_weight(estado_cancelacion)}")
print(f"Paridad C[2,3]       : {paridades_cancelacion[2,3]}")
print(f"Paridades activas    : {int(np.sum(paridades_cancelacion))}")

assert paridades_cancelacion[2, 3] == 0

Peso inicial         : 2
Paridad C[2,3]       : 0
Paridades activas    : 0


## 7. Linealidad de $\theta$

La transformación $\theta$ utiliza únicamente operaciones XOR y permutaciones de índices. Por tanto, es lineal sobre el campo binario.

Debe cumplirse:

$$
\theta(A\oplus B)
=
\theta(A)\oplus\theta(B).
$$

Esta propiedad se verificará para estados aleatorios de tamaños $z=4$ y $z=8$.

In [8]:
# ============================================================
# VALIDACIÓN DE LINEALIDAD
# ============================================================

resultados_linealidad = []

for z in (4, 8):
    rng = np.random.default_rng(2026 + z)

    for prueba in range(20):
        estado_a = rng.integers(
            0,
            2,
            size=(5, 5, z),
            dtype=np.int64,
        )

        estado_b = rng.integers(
            0,
            2,
            size=(5, 5, z),
            dtype=np.int64,
        )

        lado_izquierdo = theta(
            np.bitwise_xor(estado_a, estado_b)
        )

        lado_derecho = np.bitwise_xor(
            theta(estado_a),
            theta(estado_b),
        )

        coincide = np.array_equal(
            lado_izquierdo,
            lado_derecho,
        )

        resultados_linealidad.append(
            {
                "z": z,
                "prueba": prueba + 1,
                "linealidad_validada": coincide,
            }
        )


df_linealidad = pd.DataFrame(
    resultados_linealidad
)

df_linealidad.groupby("z")[
    "linealidad_validada"
].all()

z
4    True
8    True
Name: linealidad_validada, dtype: bool

## 8. Comparación de difusión para $z=4$ y $z=8$

Un único bit activo produce inicialmente dos posiciones activas en $D$.

Cada posición de $D$ afecta cinco bits, por lo que el patrón local de difusión es semejante para ambos tamaños.

Sin embargo, el espacio total cambia:

$$
z=4
\Rightarrow
100\text{ bits},
$$

$$
z=8
\Rightarrow
200\text{ bits}.
$$

La siguiente tabla compara el resultado.

In [9]:
# ============================================================
# COMPARACIÓN z = 4 Y z = 8
# ============================================================

resultados_difusion = []

for z in (4, 8):
    estado = create_single_active_bit_state(
        z=z,
        x=2,
        y=3,
        k=1,
    )

    paridades = column_parities(estado)
    efecto = theta_effect(estado)
    transformado = theta(estado)

    resultados_difusion.append(
        {
            "z": z,
            "bits_estado": 25 * z,
            "peso_inicial": hamming_weight(estado),
            "paridades_activas": int(np.sum(paridades)),
            "posiciones_D_activas": int(np.sum(efecto)),
            "peso_despues_theta": hamming_weight(transformado),
        }
    )


df_difusion = pd.DataFrame(
    resultados_difusion
)

df_difusion

,z,bits_estado,peso_inicial,paridades_activas,posiciones_D_activas,peso_despues_theta
0,4,100,1,1,2,11
1,8,200,1,1,2,11


## 9. Preparación para la formulación MILP

Para representar $\theta$ en MILP se necesitan tres grupos de variables:

### Variables del estado de entrada

$$
A[x,y,k]\in\{0,1\}.
$$

### Variables de paridad

$$
C[x,k]\in\{0,1\}.
$$

### Variables del efecto de difusión

$$
D[x,k]\in\{0,1\}.
$$

### Variables del estado de salida

$$
T[x,y,k]\in\{0,1\}.
$$

Las operaciones XOR no pueden escribirse directamente como igualdades lineales ordinarias.

Por ejemplo:

$$
c=a\oplus b
$$

puede representarse mediante una variable auxiliar entera $q$:

$$
a+b=c+2q.
$$

Para cinco entradas:

$$
a_0+a_1+a_2+a_3+a_4
=
c+2q,
$$

donde:

$$
q\in\{0,1,2\}.
$$

Esta formulación conserva exactamente la paridad.

## 10. Conclusiones

Se validó que:

1. la implementación utiliza un estado de forma $5\times5\times z$;
2. las paridades de columna se calculan correctamente;
3. la matriz $D$ respeta la rotación en la dimensión $k$;
4. un bit activo produce el patrón de difusión esperado;
5. dos bits activos pueden cancelarse mediante XOR;
6. $\theta$ es lineal;
7. la implementación funciona para $z=4$ y $z=8$.

La siguiente etapa será construir la representación MILP exacta de $\theta$ mediante variables binarias y variables auxiliares de paridad.

In [10]:
# ============================================================
# CONTROL FINAL
# ============================================================

assert hamming_weight(estado_inicial) == 1
assert int(np.sum(paridades)) == 1
assert int(np.sum(efecto)) == 2
assert hamming_weight(estado_theta) == 11
assert df_linealidad["linealidad_validada"].all()

print("=" * 70)
print("CAPA THETA FUNCIONAL VALIDADA")
print("=" * 70)
print("El proyecto está listo para formular theta mediante MILP.")
print("=" * 70)

CAPA THETA FUNCIONAL VALIDADA
El proyecto está listo para formular theta mediante MILP.
